# PyTorch XLM-RoBERTa base

I will fine-tune `FacebookAI/xlm-roberta-base` with its matching tokenizer for three-way natural-language inference.

Reusable loading, splitting, batching, training, and evaluation code lives in `scripts/watson_model_helpers.py`.

The premise-grouped split evaluates generalization to unseen premises.

## Venv and requirements

Different Python versions were used during development depending on hardware and software constraints. Please see the repository’s [setup documentation](../README.md#setup) for details.

## Model and tokenizer choice

- starting from general pretrained [XLM-RoBERTa base checkpoint](https://huggingface.co/FacebookAI/xlm-roberta-base)
- newly initialized three-label classification head. 
- The tokenizer receives premise first, hypothesis second, with dynamic batch padding and configurable truncation.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "scripts" / "watson_dataset_config.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from inside the project repository")
if str(PROJECT_ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

from watson_dataset_helpers import load_datasets
from watson_model_helpers import (
    split_training_data,
    load_model_and_tokenizer,
    tokenize_data,
    make_loaders,
    fine_tune,
)

## Bring in the data

In [2]:
datasets = load_datasets()

training_data = datasets["train"].copy()
test_data = datasets["test"].copy()

Kaggle files already present; using: (project_root)/data/raw


## Training-data preparation and validation split

### Goals in Preparing Training/Validation split:
- No exact premise appears in both training and validation. All rows sharing a premise stay together.
- Have roughly the same label proportions in validation and the training side as a whole.

Note: Proportions are approximate because premise groups stay intact.

In [3]:
training, validation = split_training_data(training_data)

## Data Cleaning

Prior exploration in `01_dataset_exploration.ipynb` found

- no missing values
- no duplicate premise–hypothesis pairs

No further text cleaning is planned; the original premise and hypothesis text will be kept.

## Bring in Pre-Trained Model

### XLM-Roberta Base
- XLM-Roberta Base is chosen as the pre-trained model.
- XLM-Roberta seems like a natural first check.
- Base model chosen to baseline performance using, and acknowledging, local HW constraints
- Other models may be used in the future, either in this notebook or separate notebooks

In [4]:
model, tokenizer = load_model_and_tokenizer()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Tokenization

In [5]:
training_encodings = tokenize_data(training, tokenizer)
validation_encodings = tokenize_data(validation, tokenizer)

## Batching, Padding, Training Settings

In [6]:
# Initial settings; not tuned.
batch_size = 4  # sweet spot on my HW
learning_rate = 2e-5
epochs = 1

training_loader, validation_loader = make_loaders(
    training_encodings, validation_encodings, tokenizer, batch_size
)

## Fine-tuning

In [7]:
fine_tune(model, training_loader, learning_rate, epochs)

2026-09-14 18:50:27 | Processing batch 0/2424
2026-09-14 18:50:52 | Processing batch 100/2424
2026-09-14 18:51:15 | Processing batch 200/2424
2026-09-14 18:51:39 | Processing batch 300/2424
2026-09-14 18:52:03 | Processing batch 400/2424
2026-09-14 18:52:27 | Processing batch 500/2424
2026-09-14 18:52:50 | Processing batch 600/2424
2026-09-14 18:53:15 | Processing batch 700/2424
2026-09-14 18:53:38 | Processing batch 800/2424
2026-09-14 18:54:01 | Processing batch 900/2424
2026-09-14 18:54:25 | Processing batch 1000/2424
2026-09-14 18:54:49 | Processing batch 1100/2424
2026-09-14 18:55:12 | Processing batch 1200/2424
2026-09-14 18:55:36 | Processing batch 1300/2424
2026-09-14 18:56:01 | Processing batch 1400/2424
2026-09-14 18:56:24 | Processing batch 1500/2424
2026-09-14 18:56:47 | Processing batch 1600/2424
2026-09-14 18:57:11 | Processing batch 1700/2424
2026-09-14 18:57:34 | Processing batch 1800/2424
2026-09-14 18:57:58 | Processing batch 1900/2424
2026-09-14 18:58:22 | Processing

### GPU memory investigation

Apple MPS out-of-memory issues were encountered during development. Please visit [MPS memory investigation](../docs/investigations/mps-memory.md) for the issues encountered and their resolutions. Please also visit [runtime performance documentation](../docs/runtimes/README.md) for performance findings from the MPS memory investigation.

## Evaluation

## Next observations
